# C1 — Dự báo Doanh số Q2/2026
**DATA EXPLORERS 2026 | Xe đạp Thống Nhất**

Dự báo tổng doanh số tháng 4, 5, 6/2026 bằng 2 mô hình:
- **Prophet** (Facebook) — mô hình time-series chuyên cho dữ liệu có mùa vụ và ngày lễ
- **SARIMAX** — mô hình thống kê cổ điển, tự động chọn tham số (auto_arima)
- **Ensemble** — kết hợp Prophet 60% + SARIMAX 40%

**Thứ tự chạy:** chạy tuần tự từ trên xuống dưới (Run All)


## 1. Thư viện & Cài đặt

In [ ]:
# thư viện
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
import pmdarima as pm
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

## 2. Tải & Làm sạch Dữ liệu

In [ ]:
# tải dữ liệu lịch sử (T1/2025 – T3/2026, 15 tháng)
df = pd.read_csv('../tnbike_data.csv', parse_dates=['ds'])
print(f'Dữ liệu lịch sử: {df.shape[0]} tháng')
print(f'Khoảng thời gian: {df.ds.min().strftime("%m/%Y")} – {df.ds.max().strftime("%m/%Y")}')
df.head()

In [ ]:
# tải tương lai Q2/2026
future_df = pd.read_csv('../future.csv', parse_dates=['ds'])
print('Kỳ dự báo:')
future_df

In [ ]:
# kiểm tra missing values
print('Missing values:')
print(df.isnull().sum())
print(f'\nKiểu dữ liệu:')
print(df.dtypes)

In [ ]:
# chuẩn hóa: đổi tên cột cho Prophet
df = df.rename(columns={'revenue': 'y'})
df['ds'] = pd.to_datetime(df['ds'])

# thống kê mô tả
print('Thống kê doanh số (tỷ VND):')
print((df['y']/1e9).describe().round(2))

## 3. Khám phá Dữ liệu (EDA)

In [ ]:
# biểu đồ tổng quan doanh số theo tháng
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(df.ds, df.y/1e9, marker='o', linewidth=2.5)
ax1.fill_between(df.ds, df.y/1e9, alpha=0.15)
ax1.axvspan(pd.Timestamp('2025-01-01'), pd.Timestamp('2025-03-31'),
            alpha=0.1, label='Tết 2025')
ax1.set_title('Doanh số Tháng — Xe đạp Thống Nhất (T1/2025 – T3/2026)',
              fontsize=13, fontweight='bold')
ax1.set_ylabel('Doanh số (tỷ VND)')
ax1.legend()
ax1.grid(True, alpha=0.3)

if 'qty' in df.columns:
    ax2.bar(df.ds, df.qty, alpha=0.75, width=20)
    ax2.set_ylabel('Số lượng xe (chiếc)')
else:
    ax2.bar(df.ds, df.y/1e9, alpha=0.75, width=20)
    ax2.set_ylabel('Doanh số (tỷ VND)')
ax2.set_xlabel('Tháng')
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
# phân tích xu hướng và mùa vụ
decomp = seasonal_decompose(df.set_index('ds')['y'], model='additive', period=3)
fig = decomp.plot()
fig.set_size_inches(14, 10)
fig.suptitle('Phân tích Xu hướng & Mùa vụ — Doanh số Thống Nhất Bike', fontsize=13)
fig.tight_layout()
plt.show()

## 4. Kiểm định Tính dừng (ADF Test)

In [ ]:
# ADF test — chuỗi gốc
pvalue = adfuller(df['y'].dropna())[1]
if pvalue < 0.05:
    print(f'✅ Chuỗi DỪNG — p-value = {pvalue:.4f}')
else:
    print(f'⚠️  Chuỗi KHÔNG DỪNG — p-value = {pvalue:.4f}')
    print('   → Cần sai phân bậc 1')

In [ ]:
# ADF test — sau sai phân bậc 1
pvalue_d = adfuller(df['y'].diff().dropna())[1]
if pvalue_d < 0.05:
    print(f'✅ DỪNG sau sai phân bậc 1 — p-value = {pvalue_d:.4f}')
else:
    print(f'❌ VẪN KHÔNG DỪNG — p-value = {pvalue_d:.4f}')

## 5. Chuẩn bị Dữ liệu cho Mô hình

In [ ]:
# gộp lịch sử + Q2/2026 vào cùng dataframe
future_df = future_df.rename(columns={'revenue': 'y'}) if 'revenue' in future_df.columns else future_df
df_full_model = pd.concat([df, future_df], ignore_index=True)
df_full_model['ds'] = pd.to_datetime(df_full_model['ds'])

# tách train / predict
training  = df_full_model.iloc[:-3, :].copy()
forecast_period = df_full_model.iloc[-3:, :].copy()
print(f'Train: {len(training)} tháng | Dự báo: {len(forecast_period)} tháng')
print(f'Train cuối: {training.ds.max().strftime("%m/%Y")}')
print(f'Dự báo từ: {forecast_period.ds.min().strftime("%m/%Y")} đến {forecast_period.ds.max().strftime("%m/%Y")}')

In [ ]:
# ngày lễ: Tết Nguyên Đán
dates_tet = pd.to_datetime(df_full_model[df_full_model.is_tet == 1].ds)
tet = pd.DataFrame({
    'holiday': 'tet_nguyen_dan',
    'ds':       dates_tet,
    'lower_window': -7,
    'upper_window':  7
})

# ngày lễ: Thiếu nhi 1/6
dates_thieu_nhi = pd.to_datetime(df_full_model[df_full_model.is_thieu_nhi == 1].ds)
thieu_nhi = pd.DataFrame({
    'holiday': 'ngay_thieu_nhi',
    'ds':       dates_thieu_nhi,
    'lower_window': -14,
    'upper_window':   3
})

holidays = pd.concat([tet, thieu_nhi])
print(f'Ngày lễ đã khai báo: {holidays.holiday.nunique()} loại, {len(holidays)} điểm')
holidays

In [ ]:
# bỏ cột is_tet, is_thieu_nhi khỏi training (Prophet không cần)
drop_cols = [c for c in ['is_tet', 'is_thieu_nhi'] if c in training.columns]
training = training.drop(columns=drop_cols)

## 6. Mô hình Prophet

In [ ]:
# lấy tham số tốt nhất từ Parameter Tuning
parameters = pd.read_csv('../Forecasting Product/best_params_prophet.csv', index_col=0)
changepoint_prior_scale = float(parameters.loc['changepoint_prior_scale'][0])
holidays_prior_scale    = float(parameters.loc['holidays_prior_scale'][0])
seasonality_prior_scale = float(parameters.loc['seasonality_prior_scale'][0])
seasonality_mode        = parameters.loc['seasonality_mode'][0]
print('Tham số Prophet:')
parameters

In [ ]:
# huấn luyện Prophet
m = Prophet(
    holidays               = holidays,
    seasonality_mode       = seasonality_mode,
    seasonality_prior_scale= seasonality_prior_scale,
    holidays_prior_scale   = holidays_prior_scale,
    changepoint_prior_scale= changepoint_prior_scale,
    yearly_seasonality     = False,
    weekly_seasonality     = False,
    daily_seasonality      = False
)
m.add_seasonality(name='monthly', period=30.5, fourier_order=5)
m.fit(training[['ds','y']])
print('✅ Prophet đã huấn luyện xong')

In [ ]:
# dự báo
future = m.make_future_dataframe(periods=3, freq='MS')
forecast_prophet = m.predict(future)

# kết quả Q2/2026
predictions_prophet = forecast_prophet[['ds','yhat','yhat_lower','yhat_upper']].tail(3).reset_index(drop=True)
predictions_prophet['tháng'] = predictions_prophet.ds.dt.strftime('T%m/%Y')
print('Dự báo Prophet Q2/2026:')
for _, r in predictions_prophet.iterrows():
    print(f'  {r.tháng}: {r.yhat/1e9:.2f}T VND  [{r.yhat_lower/1e9:.2f}T – {r.yhat_upper/1e9:.2f}T]')

In [ ]:
# biểu đồ thành phần Prophet (xu hướng + mùa vụ + ngày lễ)
fig = m.plot_components(forecast_prophet)
fig.suptitle('Thành phần Mô hình Prophet', fontsize=13, fontweight='bold', y=1.01)
fig.tight_layout()
plt.show()

## 7. Đánh giá Prophet (Cross-Validation)

In [ ]:
# kiểm định chéo: 3 tháng horizon
df_cv = cross_validation(m, horizon='90 days', period='30 days', initial='90 days')
metrics = performance_metrics(df_cv)
rmse = metrics['rmse'].mean()
mape = metrics['mape'].mean()
print(f'Prophet Cross-Validation:')
print(f'  RMSE : {rmse:,.0f} VND')
print(f'  MAPE : {mape*100:.1f}%')
metrics.head()

In [ ]:
# sai số theo thời gian
fig = plot_cross_validation_metric(df_cv, metric='rmse')
plt.suptitle('RMSE Prophet theo Horizon', fontsize=12, fontweight='bold')
fig.tight_layout()
plt.show()

## 8. Mô hình SARIMAX

In [ ]:
# lấy tham số tốt nhất từ Parameter Tuning
params_s = pd.read_csv('../Forecasting Product/best_params_sarimax.csv', index_col=0)
p = int(params_s.loc['p'][0])
d = int(params_s.loc['d'][0])
q = int(params_s.loc['q'][0])
P = int(params_s.loc['P'][0])
D = int(params_s.loc['D'][0])
Q = int(params_s.loc['Q'][0])
m_s = int(params_s.loc['m'][0])
print(f'Tham số SARIMAX: order=({p},{d},{q}) × seasonal=({P},{D},{Q},{m_s})')

In [ ]:
# huấn luyện SARIMAX
model_s = pm.ARIMA(
    order          = (p, d, q),
    seasonal_order = (P, D, Q, m_s),
    suppress_warnings=True
)
model_s.fit(training['y'].values)
print('✅ SARIMAX đã huấn luyện xong')
print(model_s.summary())

In [ ]:
# dự báo Q2/2026
sarimax_vals = model_s.predict(n_periods=3)
months_q2 = pd.date_range('2026-04-01', periods=3, freq='MS')
predictions_sarimax = pd.DataFrame({
    'ds':   months_q2,
    'yhat': sarimax_vals
})
print('Dự báo SARIMAX Q2/2026:')
for _, r in predictions_sarimax.iterrows():
    print(f"  {r.ds.strftime('T%m/%Y')}: {r.yhat/1e9:.2f}T VND")

In [ ]:
# biểu đồ SARIMAX: fitted vs actual
fitted_vals = model_s.predict_in_sample()
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(training.ds, training.y/1e9, marker='o', linewidth=2, label='Thực tế')
ax.plot(training.ds, fitted_vals/1e9, linestyle='--', linewidth=2, label='Fitted SARIMAX')
ax.set_title('SARIMAX — Khớp mô hình trên tập huấn luyện', fontsize=13, fontweight='bold')
ax.set_ylabel('Doanh số (tỷ VND)')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 9. Ensemble: Prophet 60% + SARIMAX 40%

In [ ]:
# kết hợp hai mô hình
prophet_q2 = predictions_prophet['yhat'].values
sarimax_q2 = predictions_sarimax['yhat'].values

ensemble_yhat = 0.6 * prophet_q2 + 0.4 * sarimax_q2

ensemble = pd.DataFrame({
    'ds':          months_q2,
    'yhat':        ensemble_yhat,
    'yhat_lower':  predictions_prophet['yhat_lower'].values,
    'yhat_upper':  predictions_prophet['yhat_upper'].values,
    'prophet':     prophet_q2,
    'sarimax':     sarimax_q2
})
print('Dự báo Ensemble Q2/2026:')
for _, r in ensemble.iterrows():
    print(f"  {r.ds.strftime('T%m/%Y')}: {r.yhat/1e9:.2f}T VND")

## 10. Trực quan hóa Tổng hợp

In [ ]:
# so sánh 3 mô hình — bar chart
months_label = ['T4/2026', 'T5/2026', 'T6/2026']
x = np.arange(3)
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
b1 = ax.bar(x - width, prophet_q2/1e9,  width, label='Prophet',          alpha=0.85)
b2 = ax.bar(x,          sarimax_q2/1e9,  width, label='SARIMAX',           alpha=0.85)
b3 = ax.bar(x + width,  ensemble_yhat/1e9, width, label='Ensemble (60/40)', alpha=0.85)

for bars, vals in zip([b1, b2, b3], [prophet_q2, sarimax_q2, ensemble_yhat]):
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.1,
                f'{val/1e9:.1f}T',
                ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(months_label, fontsize=12)
ax.set_ylabel('Doanh số (tỷ VND)')
ax.set_title('So sánh Dự báo: Prophet vs SARIMAX vs Ensemble — Q2/2026',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
fig.tight_layout()
plt.show()

In [ ]:
# biểu đồ đường — lịch sử + dự báo ensemble
fig, ax = plt.subplots(figsize=(14, 6))

# lịch sử
ax.plot(training.ds, training.y/1e9, 'o-', linewidth=2.5, label='Lịch sử', zorder=5)

# dự báo từng mô hình
ax.plot(months_q2, prophet_q2/1e9,  's--', linewidth=2, label='Prophet',  markersize=9)
ax.plot(months_q2, sarimax_q2/1e9,  '^--', linewidth=2, label='SARIMAX',  markersize=9)
ax.plot(months_q2, ensemble_yhat/1e9, 'D-', linewidth=3, label='Ensemble', markersize=10, zorder=6)

# khoảng tin cậy 95%
ax.fill_between(months_q2,
                ensemble['yhat_lower']/1e9,
                ensemble['yhat_upper']/1e9,
                alpha=0.15, label='KTC 95%')

# đường phân cách lịch sử / dự báo
ax.axvline(pd.Timestamp('2026-03-31'), linestyle=':', alpha=0.5, label='Ranh giới dự báo')

# nhãn giá trị ensemble
for d, v in zip(months_q2, ensemble_yhat):
    ax.annotate(f'{v/1e9:.1f}T', (d, v/1e9),
                textcoords='offset points', xytext=(0, 12),
                fontsize=10, fontweight='bold', ha='center')

ax.set_title('Ensemble Dự báo Doanh số Q2/2026 — Xe đạp Thống Nhất',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Doanh số (tỷ VND)')
ax.set_xlabel('Tháng')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 11. Xuất Kết quả

In [ ]:
# lưu predictions để dùng cho Ensemble chéo
import os
os.makedirs('../Forecasting Product/Ensemble', exist_ok=True)

predictions_prophet.to_csv('../Forecasting Product/Ensemble/predictions_prophet.csv', index=False)
predictions_sarimax.to_csv('../Forecasting Product/Ensemble/predictions_sarimax.csv', index=False)
ensemble.to_csv('../Forecasting Product/Ensemble/predictions_ensemble_final.csv', index=False)

print('✅ Đã lưu:')
print('  - predictions_prophet.csv')
print('  - predictions_sarimax.csv')
print('  - predictions_ensemble_final.csv')

In [ ]:
# tóm tắt kết quả
print('=' * 55)
print('    DỰ BÁO DOANH SỐ Q2/2026 — XE ĐẠP THỐNG NHẤT')
print('=' * 55)
for _, r in ensemble.iterrows():
    tg = r.ds.strftime('Tháng %m/%Y')
    print(f'  {tg}: {r.yhat/1e9:.2f} tỷ VND'
          f'  (Prophet {r.prophet/1e9:.2f}T | SARIMAX {r.sarimax/1e9:.2f}T)')
print('-' * 55)
print(f'  Tổng Q2/2026  : {ensemble.yhat.sum()/1e9:.2f} tỷ VND')
print(f'  TB/tháng      : {ensemble.yhat.mean()/1e9:.2f} tỷ VND')
print('=' * 55)